# Description

In this notebook, I will explore the benchmark Human Eval and using GPT4 to generate it.

In [1]:
import os 
import sys
import numpy as np 
import pandas as pd 
import re
import io
import re
import ast
import types
import unittest
import importlib
from typing import List, Tuple, Dict, Any, Set
import load_dotenv
from openai import OpenAI

# 1. Load data

In [2]:
PATH_CSV_DATA = "data/human_eval.csv"

In [3]:
load_dotenv.load_dotenv()

# OPEN_AI_API = os.getenv("OPEN_AI_API")
OPEN_AI_API = os.getenv("OPEN_AI_API_v2")
if OPEN_AI_API is None:
    raise ValueError("OPEN_AI_API environment variable not set")
else:
    print("API key loaded successfully")

client = OpenAI(api_key=OPEN_AI_API)

API key loaded successfully


In [4]:
df = pd.read_csv(PATH_CSV_DATA)
print(f"Dataframe shape: {df.shape}")
df.sample(1)

Dataframe shape: (164, 5)


,task_id,prompt,canonical_solution,test,entry_point
40,HumanEval/40,"\n\ndef triples_sum_to_zero(l: list):\n """"""...",for i in range(len(l)):\n for j in ...,\n\nMETADATA = {}\n\n\ndef check(candidate):\n...,triples_sum_to_zero


In [5]:
idx = np.random.randint(0, df.shape[0])

code_description = df.loc[idx, "prompt"]
test_case = df.loc[idx, "test"]
entry_point = df.loc[idx, "entry_point"]

print("Code description:")
print(code_description)
print("=" * 20)
print("Test Case:")
print(test_case)

Code description:

def even_odd_palindrome(n):
    """
    Given a positive integer n, return a tuple that has the number of even and odd
    integer palindromes that fall within the range(1, n), inclusive.

    Example 1:

        Input: 3
        Output: (1, 2)
        Explanation:
        Integer palindrome are 1, 2, 3. one of them is even, and two of them are odd.

    Example 2:

        Input: 12
        Output: (4, 6)
        Explanation:
        Integer palindrome are 1, 2, 3, 4, 5, 6, 7, 8, 9, 11. four of them are even, and 6 of them are odd.

    Note:
        1. 1 <= n <= 10^3
        2. returned tuple has the number of even and odd integer palindromes respectively.
    """

Test Case:
def check(candidate):

    # Check some simple cases
    assert candidate(123) == (8, 13)
    assert candidate(12) == (4, 6)
    assert candidate(3) == (1, 2)
    assert candidate(63) == (6, 8)
    assert candidate(25) == (5, 6)
    assert candidate(19) == (4, 6)
    assert candidate(9) == (4,

# 2. Using GPT4 to generate sample

## 2.1. Generate code

In [6]:
def extract_function(llm_text):
    # 1) Grab text between <code>...</code>
    m = re.search(r"<code>\s*(.*?)\s*</code>", llm_text, flags=re.S|re.M)
    if not m:
        raise ValueError("No <code> block found")
    code = m.group(1)

    # 2) Optionally, if the model sometimes adds backticks, strip them
    code = re.sub(r"^```(?:python)?\s*|\s*```$", "", code.strip())

    return code

def extract_reasoning(llm_text):
    # 1) Grab text between <reasoning>...</reasoning>
    m = re.search(r"<reasoning>\s*(.*?)\s*</reasoning>", llm_text, flags=re.S|re.M)
    if not m:
        raise ValueError("No <reasoning> block found")
    reasoning = m.group(1).strip()
    return reasoning

In [7]:
COT = """
You are a careful reasoning coding assistant. When you receive a question or problem, you must:
1. Understand what is being asked.
2. Reason through each step slowly and logically.
3. Give the final answer.
"""

In [8]:
constraints = """
Output only a complete and valid Python code for this function. Do not change the provided function signature.
Wrap your output strictly between the markers:
<code>
... your code ...
</code>
"""

COT = """
Before giving the final code, you MUST think step-by-step and show your reasoning.
Explain:
1. what the function must do  
2. possible edge cases  
3. the algorithm you will implement  
4. why this algorithm is correct  
5. then produce the final code

After reasoning, output the final answer strictly in this format:
<reasoning>
(Your full reasoning here)
</reasoning>
"""


input_prompt = f"""write a complete python function
based on the following description:\n{code_description}.\n
{COT}.
with the following constraints:\n{constraints}
"""

print("Input prompt to GPT-4:")
print(input_prompt)

Input prompt to GPT-4:
write a complete python function
based on the following description:

def even_odd_palindrome(n):
    """
    Given a positive integer n, return a tuple that has the number of even and odd
    integer palindromes that fall within the range(1, n), inclusive.

    Example 1:

        Input: 3
        Output: (1, 2)
        Explanation:
        Integer palindrome are 1, 2, 3. one of them is even, and two of them are odd.

    Example 2:

        Input: 12
        Output: (4, 6)
        Explanation:
        Integer palindrome are 1, 2, 3, 4, 5, 6, 7, 8, 9, 11. four of them are even, and 6 of them are odd.

    Note:
        1. 1 <= n <= 10^3
        2. returned tuple has the number of even and odd integer palindromes respectively.
    """
.


Before giving the final code, you MUST think step-by-step and show your reasoning.
Explain:
1. what the function must do  
2. possible edge cases  
3. the algorithm you will implement  
4. why this algorithm is correct  
5. then

In [9]:
response = client.chat.completions.create(
    model="gpt-4o",  # or "gpt-4o" 
    messages=[
        {"role": "system", "content": "You are an expert in Python."},
        {"role": "user", "content": input_prompt},
    ],
)

output = response.choices[0].message.content

print("Response from OpenAI:")
print(output)

Response from OpenAI:
<reasoning>
1. The function `even_odd_palindrome(n)` is required to return a tuple indicating how many even and how many odd integer palindromes exist between 1 and `n`, inclusive. A palindrome is an integer that reads the same backward as forward.

2. Edge cases to consider include:
   - `n = 1`: The smallest possible input, where the function should return (0, 1) because only 1 is a palindrome and it is odd.
   - If `n` is within the given constraint, i.e., `1 <= n <= 10^3`, our function should handle all such values smoothly.
   - Since no input error handling is needed under these constraints, it's assumed `n` will always be a positive integer within the range.

3. Algorithm:
   - Initialize two counters, `even_count` and `odd_count`, to zero.
   - Iterate through each number `i` from 1 to `n`.
   - Check if `i` is a palindrome by converting it to a string and comparing the string to its reverse.
   - If `i` is a palindrome, determine if it is even or odd.
   

We can extract the complete code

In [10]:
completed_code = extract_function(output)
print(f"The complete code:\n")
print(completed_code)

The complete code:

def even_odd_palindrome(n):
    """
    Given a positive integer n, return a tuple that has the number of even and odd
    integer palindromes that fall within the range(1, n), inclusive.
    """
    even_count = 0
    odd_count = 0

    for i in range(1, n + 1):
        str_i = str(i)
        if str_i == str_i[::-1]:  # Checking if the number is a palindrome
            if i % 2 == 0:
                even_count += 1
            else:
                odd_count += 1

    return (even_count, odd_count)


In [11]:
reasoning_text = extract_reasoning(output)
print(f"The reasoning:\n")
print(reasoning_text)

The reasoning:

1. The function `even_odd_palindrome(n)` is required to return a tuple indicating how many even and how many odd integer palindromes exist between 1 and `n`, inclusive. A palindrome is an integer that reads the same backward as forward.

2. Edge cases to consider include:
   - `n = 1`: The smallest possible input, where the function should return (0, 1) because only 1 is a palindrome and it is odd.
   - If `n` is within the given constraint, i.e., `1 <= n <= 10^3`, our function should handle all such values smoothly.
   - Since no input error handling is needed under these constraints, it's assumed `n` will always be a positive integer within the range.

3. Algorithm:
   - Initialize two counters, `even_count` and `odd_count`, to zero.
   - Iterate through each number `i` from 1 to `n`.
   - Check if `i` is a palindrome by converting it to a string and comparing the string to its reverse.
   - If `i` is a palindrome, determine if it is even or odd.
     - Increment `eve

## 2.2. Evaluate the generated code

In [12]:
import builtins
import typing

def create_namespace():
    ns = {}

    # 1. Standard builtins (print, len, etc.)
    ns.update({k: getattr(builtins, k) for k in dir(builtins)})

    # 2. Install common typing names (List, Optional, etc.)
    for name in typing.__all__:
        ns[name] = getattr(typing, name)

    # 3. (Optional) Add math, random, itertools, etc.
    import math, random, itertools, statistics
    ns.update({
        'math': math,
        'random': random,
        'itertools': itertools,
        'statistics': statistics,
    })

    return ns

In [13]:
def evaluate_asserts(generated_code: str, test_code: str, entry_point: str):
    # ns = {}
    ns = create_namespace()
    
    # 1. Exec both code strings
    exec(generated_code, ns)
    exec(test_code, ns)

    candidate = ns[entry_point]     # the model's function
    check_fn = ns["check"]          # original check() function
    
    # 2. Parse the test code AST
    tree = ast.parse(test_code)

    # 3. Find the check() function body
    check_body = None
    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name == "check":
            check_body = node.body
            break

    if check_body is None:
        raise ValueError("check() function not found.")
    
    # 4. Evaluate each assert individually
    results = []
    for idx, stmt in enumerate(check_body):
        if isinstance(stmt, ast.Assert):
            # Convert AST back to executable code
            code = compile(ast.Module([stmt], type_ignores=[]), "<assert>", "exec")
            try:
                exec(code, {**ns, "candidate": candidate})
                results.append(("pass", None))
            except Exception as e:
                results.append(("fail", repr(e)))

    # 5. Compute pass percentage
    total = len(results)
    passed = sum(1 for r, _ in results if r == "pass")
    percentage = passed / total if total > 0 else 0.0

    return {
        "total_asserts": total,
        "passed": passed,
        "percentage": percentage,
        "detail": results
    }

In [14]:
result = evaluate_asserts(completed_code, test_case, entry_point)
print(result)

{'total_asserts': 8, 'passed': 8, 'percentage': 1.0, 'detail': [('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None)]}


# 3. Run through all sample

In [15]:
list_df = []

for idx in range(df.shape[0]):
    if idx % 10 == 0:
        print(f"Processing idx={idx}/{df.shape[0]}")
    
    # 1. Prepare input prompt
    code_description = df.loc[idx, "prompt"]
    test_case = df.loc[idx, "test"]
    entry_point = df.loc[idx, "entry_point"]

    input_prompt = f"""write a complete python function
    based on the following description:\n{code_description}.\n
    {COT}.
    with the following constraints:\n{constraints}
    """

    # 2. Generate code with GPT-4o
    response = client.chat.completions.create(
        model="gpt-4o",  # or "gpt-4o" 
        messages=[
            {"role": "system", "content": "You are an expert in Python."},
            {"role": "user", "content": input_prompt},
        ],
    )

    output = response.choices[0].message.content
    completed_code = extract_function(output)
    reasoning_text = extract_reasoning(output)
    
    # 3. Evaluate the generated code
    result = evaluate_asserts(completed_code, test_case, entry_point)
    total_asserts = result["total_asserts"]
    passed_asserts = result["passed"]
    percentage = result["percentage"]
    
    list_df.append({
        "description": code_description,
        "generated_code": completed_code,
        "test_case": test_case,
        "entry_point": entry_point,
        "total_asserts": total_asserts,
        "passed_asserts": passed_asserts,
        "percentage": percentage,
        "reasoning_text": reasoning_text,
    })

Processing idx=0/164
Processing idx=10/164
Processing idx=20/164
Processing idx=30/164
Processing idx=40/164
Processing idx=50/164
Processing idx=60/164
Processing idx=70/164
Processing idx=80/164
Processing idx=90/164
Processing idx=100/164
Processing idx=110/164
Processing idx=120/164
Processing idx=130/164
Processing idx=140/164
Processing idx=150/164
Processing idx=160/164


In [16]:
output_df = pd.DataFrame(list_df)
print(f'Output dataframe shape: {output_df.shape}')
output_df.sample()

Output dataframe shape: (164, 8)


,description,generated_code,test_case,entry_point,total_asserts,passed_asserts,percentage,reasoning_text
23,"\n\ndef strlen(string: str) -> int:\n """""" R...","def strlen(string: str) -> int:\n """"""Return...","\n\nMETADATA = {\n 'author': 'jt',\n 'da...",strlen,3,3,1.0,1. What the function must do:\nThe function `s...


In [17]:
# Save to CSV
output_df.to_csv("data/human_eval_generated_gpt4o_COT.csv", index=False)

## 3.1. Check generated code

In [18]:
average_percentage = output_df["percentage"].mean()
print(f"Average pass percentage over all samples: {average_percentage:.2%}")    

Average pass percentage over all samples: 93.80%


In [19]:
total_num_passed = output_df["passed_asserts"].sum()
print(f"Total number of passed asserts: {total_num_passed}")

Total number of passed asserts: 1125
